# Talking to models with the OpenAI-style chat format

This notebook looks closely at the **request** we send to a chat model and the **response** we get back. WebLLM uses almost the same format as OpenAI's **Chat Completions API** (an **API** is a set of rules for how one program talks to another). So many tools copied this format that it became a standard, so what you learn here carries over to many other AI tools.

Everything here runs in your browser.

## Step 1: Get set up

Small cells that bring in the Python tools we need, check that your browser can run models, load the WebLLM library, and set up the model loader.

### 1a: Import tools

- `json` turns Python data into text and back. We use it to hand data to JavaScript.
- `pyjs` connects Python to the browser's **JavaScript**, the language web pages run on. WebLLM is written in JavaScript, so we need this bridge.
- `time` reads the clock, so we can measure how fast the model writes (Step 9).
- `pprint` ("pretty print") prints big nested data neatly, one item per line.
- `display` lets a cell show something and then update it in place (we use it for a progress bar).

In [ ]:
import json
import pyjs
import time
from pprint import pprint
from IPython.display import display

js = pyjs.js  # a doorway from Python into the browser's JavaScript

print("Tools imported.")

### 1b: Check for WebGPU

Models run on your computer's graphics chip (GPU). Browsers reach the GPU through a feature called **WebGPU**. Recent Chrome and Edge on a laptop have it; many phones and older browsers do not.

In [ ]:
if getattr(js.navigator, "gpu", None) is not None:
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

### 1c: Load WebLLM

**WebLLM** is a free JavaScript library that runs language models inside the browser. This line fetches it from the internet (it takes a few seconds the first time).

In [ ]:
webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

### 1d: Translate between Python and JavaScript

WebLLM runs in JavaScript, and JavaScript cannot read Python dictionaries directly. Both languages understand **JSON**, a plain-text format for data, so we translate by way of JSON text:

- `js_obj()` turns a Python dictionary into a JavaScript object, for requests we send to WebLLM.
- `py_obj()` goes the other way, turning a JavaScript object (like the model's reply) into Python dictionaries and lists.

You met `js_obj()` in Step 7c of the starter notebook.

In [ ]:
def js_obj(d):
    """Turn a Python dict into a JavaScript object (by way of JSON text)."""
    return js.JSON.parse(json.dumps(d))


def py_obj(value):
    """Turn a JavaScript object into Python dicts and lists (by way of JSON text)."""
    return json.loads(js.JSON.stringify(value))


print("Translators ready.")

### 1e: Set up the model loader

- `engine` is the part of WebLLM that actually runs a model. It starts as `None` (Python for "nothing yet") and gets created the first time we need it.
- `loaded_model` remembers the name of the model that is currently loaded, so later steps can check that one is ready.
- `progress_bar()` shows a bar that fills up while a model downloads or loads.
- `start_model()` gets a model ready to run. If the model is not saved in your browser yet, it downloads it first.

This is the same loader as in the starter notebook. You do not need to follow every line. Just run it.

In [ ]:
engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    # JavaScript needs its own handle on the Python function to call it
    js_show, handle = pyjs.create_callable(show)
    return status, js_show, handle


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress, handle = progress_bar()

    if engine is None:
        engine = webllm.MLCEngine.new()   # create an empty engine
    engine.setInitProgressCallback(progress)

    try:
        await engine.reload(model_id)
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await engine.reload(model_id, js_obj({"sliding_window_size": -1}))
        else:
            raise

print("Model loader is ready.")

## Step 2: Load a model

This notebook is about the chat format, so we just load one recommended chat model: **Gemma 3 1B**. If it is not saved in your browser yet, it downloads first (about 700 MB). To try another model, paste its name from notebook 01 into `MODEL_NAME`.

In [ ]:
MODEL_NAME = "gemma3-1b-it-q4f16_1-MLC"

await start_model(MODEL_NAME)
loaded_model = MODEL_NAME
print(f"{loaded_model} is loaded and ready.")

## Step 3: Build the smallest possible chat request

A chat request starts with a **list of messages**. Each message is a Python dictionary with a `"role"` (who is speaking) and some `"content"` (what they said).

The three roles are:

- `system`: instructions that set the model's behavior for the whole conversation
- `user`: what you ask
- `assistant`: earlier replies from the model

We start with just one user message.

In [ ]:
messages = [
    {"role": "user", "content": "Explain what a token is in two short sentences."}
]

messages

## Step 4: Add the settings

Along with `messages`, we send settings (also called **arguments**) that affect how the model replies:

- **`temperature`**: how random the answer is.
- **`max_tokens`**: the most **tokens** the model may write. A token is a small chunk of text, often a short word or part of a word.
- **`frequency_penalty`**: discourages repeating the same words, which small models tend to do.

We will try each of these, and a few more, in Step 12.

In [ ]:
TEMPERATURE = 0.7
MAX_TOKENS = 300
FREQUENCY_PENALTY = 0.5

request = {
    "messages": messages,
    "temperature": TEMPERATURE,
    "max_tokens": MAX_TOKENS,
    "frequency_penalty": FREQUENCY_PENALTY,
}

request

## Step 5: Check that a model is loaded

The next steps need the model from Step 2. This cell checks.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Go back to Step 2 and run it until it says the model is ready.")
else:
    print(f"Ready to chat with {loaded_model}.")

## Step 6: Send the request

This is still a normal Python dictionary, so `js_obj()` translates it for JavaScript. Then `engine.chat.completions.create(...)` sends it to the model. The name copies OpenAI's API, which uses the same name for the same job.

`await` means "wait here until the model finishes."

In [ ]:
reply = await engine.chat.completions.create(js_obj(request))

print("Got a reply.")

## Step 7: Look at the full response

The reply contains the answer, some bookkeeping fields, and token counts. `py_obj()` turns it into Python dictionaries and lists so `pprint` can show all of it.

In [ ]:
response = py_obj(reply)
pprint(response)

## Step 8: Pull out the most important fields

The most useful parts are usually:

- `choices[0].message.content`: the answer text
- `choices[0].finish_reason`: why the model stopped. `"stop"` means it finished on its own; `"length"` means it hit `max_tokens`.
- `usage.prompt_tokens`: how many tokens you sent
- `usage.completion_tokens`: how many tokens the model wrote

In [ ]:
print("Answer:")
print(response["choices"][0]["message"]["content"])
print()

usage = response.get("usage", {})

print("finish_reason:    ", response["choices"][0].get("finish_reason"))
print("prompt_tokens:    ", usage.get("prompt_tokens"))
print("completion_tokens:", usage.get("completion_tokens"))
print("total_tokens:     ", usage.get("total_tokens"))

## Step 9: How fast is the model?

Speed is measured in **tokens per second**: how many tokens the model writes each second. A bigger number means a faster model. It depends on the model's size and on your laptop's graphics chip, so your numbers will differ from a classmate's.

### 9a: Time it yourself

We send the request from Step 4 again, and read the clock just before and just after.

- `time.time()` gives the current time in seconds.
- Subtracting the start time from the end time gives how many seconds the model took.
- Dividing the tokens it wrote (`completion_tokens`) by those seconds gives tokens per second.

In [ ]:
start = time.time()
reply = await engine.chat.completions.create(js_obj(request))
seconds = time.time() - start

timed = py_obj(reply)
tokens_written = timed["usage"]["completion_tokens"]

print(f"The model wrote {tokens_written} tokens in {seconds:.1f} seconds.")
print(f"That is about {tokens_written / seconds:.1f} tokens per second.")

### 9b: Ask WebLLM for its own numbers

WebLLM also times itself and puts the results in the reply, under `usage` then `extra`. It splits the work into two parts:

- **Reading** your prompt (called **prefill**). This is fast, because the model reads all the prompt tokens at once.
- **Writing** the answer (called **decode**). This is slower, because the model writes one token at a time.

Your number from 9a is a little lower than WebLLM's writing speed, because your clock also counted the reading time.

In [ ]:
extra = timed["usage"].get("extra")

if extra is None:
    print("This version of WebLLM did not include timing numbers in the reply.")
else:
    print("Reading speed (tokens per second):", round(extra.get("prefill_tokens_per_s", 0), 1))
    print("Writing speed (tokens per second):", round(extra.get("decode_tokens_per_s", 0), 1))
    print("Wait before the first token (sec): ", round(extra.get("time_to_first_token_s", 0), 2))

### 9c: Compare two models

Try this: change `MODEL_NAME` in Step 2 to a smaller model, such as `"SmolLM2-360M-Instruct-q4f16_1-MLC"`, and run Step 2 again. Then come back and run 9a and 9b. Is the smaller model faster? By how much?

## Step 10: Make a reusable helper

Now that you have seen the raw format, here is a helper that does Steps 4, 6 and 7 in one go. Give it a list of messages, plus any settings you want to change, and it gives back the response as Python data.

`**settings` collects any extra settings you pass in, like `temperature=0.0`, and `request.update(settings)` puts them into the request, replacing the defaults.

In [ ]:
async def chat_once(messages, **settings):
    request = {
        "messages": messages,
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
        "frequency_penalty": FREQUENCY_PENALTY,
    }
    request.update(settings)
    reply = await engine.chat.completions.create(js_obj(request))
    return py_obj(reply)


def answer_of(response):
    """The answer text inside a response."""
    return response["choices"][0]["message"]["content"]


response = await chat_once(messages)
print(answer_of(response))

## Step 11: Add a system message and earlier turns

### 11a: A system message

A `system` message goes first and sets the overall behavior. It does not guarantee perfect obedience, especially with small models, but it often helps. Compare this answer with the one in Step 8.

In [ ]:
SYSTEM = "You are a patient teacher for beginners. Use a simple example."

system_messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Explain what a token is in two short sentences."},
]

response = await chat_once(system_messages)
print(answer_of(response))

### 11b: A multi-turn conversation

A conversation with several turns is just a longer `messages` list. You keep adding what the user said and what the assistant said. **The model does not remember anything outside the list you send.** Here we make up an earlier exchange and then ask about it.

In [ ]:
conversation_messages = [
    {"role": "system", "content": "You are a patient teacher for beginners."},
    {"role": "user", "content": "My name is Maya."},
    {"role": "assistant", "content": "Nice to meet you, Maya."},
    {"role": "user", "content": "What name did I just tell you?"},
]

response = await chat_once(conversation_messages)
print(answer_of(response))

## Step 12: Experiment with the settings

Each part below changes one setting at a time. Run the cells a few times and compare the outputs.

### 12a: temperature

`temperature` controls randomness. Low values give safer, more repeatable answers. Higher values give more variety.

In [ ]:
prompt = [{"role": "user", "content": "Write a one-line slogan for a coffee shop."}]

for value in [0.0, 0.7, 1.2]:
    response = await chat_once(prompt, temperature=value)
    print(f"temperature={value}: {answer_of(response)}\n")

### 12b: top_p

`top_p` is another way to control randomness. At each step the model has a list of possible next tokens, each with a chance. `top_p` keeps only the most likely ones whose chances add up to `top_p` (so `0.3` keeps just the top few, and `1.0` keeps them all).

In [ ]:
for value in [0.3, 0.7, 1.0]:
    response = await chat_once(prompt, top_p=value)
    print(f"top_p={value}: {answer_of(response)}\n")

### 12c: max_tokens

`max_tokens` sets an upper limit on how long the answer can be. If the model hits the limit, `finish_reason` is `"length"`.

In [ ]:
tips = [{"role": "user", "content": "List five study tips for a hard class."}]

for value in [20, 60, 150]:
    response = await chat_once(tips, max_tokens=value)
    print(f"max_tokens={value}, finish_reason={response['choices'][0].get('finish_reason')}")
    print(answer_of(response))
    print()

### 12d: frequency_penalty

`frequency_penalty` pushes the model away from repeating the same words again and again. This is especially useful with small models.

In [ ]:
chant = [{"role": "user", "content": "Write a short chant about bicycles."}]

for value in [0.0, 0.5, 1.0]:
    response = await chat_once(chant, frequency_penalty=value, max_tokens=80)
    print(f"frequency_penalty={value}: {answer_of(response)}\n")

### 12e: presence_penalty

`presence_penalty` nudges the model toward bringing in new topics or words instead of staying close to what it has already said.

In [ ]:
slogans = [{"role": "user", "content": "Give me ideas for a school club slogan."}]

for value in [0.0, 0.8, 1.5]:
    response = await chat_once(slogans, presence_penalty=value, max_tokens=60)
    print(f"presence_penalty={value}: {answer_of(response)}\n")

### 12f: seed

`seed` asks the model to be repeatable: the same seed should give the same answer. Here seed 7 is used twice, so the first two answers should match. Exact repeatability is not always perfect across computers.

In [ ]:
welcome = [{"role": "user", "content": "Write a two-line welcome message for a math club."}]

for value in [7, 7, 21]:
    response = await chat_once(welcome, seed=value)
    print(f"seed={value}: {answer_of(response)}\n")

### 12g: stop

`stop` tells the model to stop as soon as it writes a marker you choose. Here the marker is `###`, so the model should stop after the first line.

In [ ]:
moon = [{"role": "user", "content": "Write three short lines about the moon. End each line with ###"}]

response = await chat_once(moon, stop=["###"], max_tokens=80)
print(answer_of(response))
print("finish_reason:", response["choices"][0].get("finish_reason"))

## Step 13: The same shape also works with cloud models

The important idea is the **shape** of the request and response. If you later use a cloud model, you will still send a `messages` list plus settings like `temperature`, and you will still get back `choices`, `message.content`, `finish_reason`, and `usage`.

WebLLM changes **where** the model runs. The data format stays familiar.

## Your turn

Change the messages and settings above. Try to make the model sound more formal, more playful, or more repetitive, and notice which setting matters most.